# 🥉 Bronze Layer: Production-Scale Raw Ingestion

This notebook provides an interactive walkthrough of the **Bronze Layer Ingestion**. We will move data from raw Parquet files (S3-Compatible) into a managed **Apache Iceberg** table registered in the **Project Nessie** catalog.

### Objectives:
1. Initialize a Spark Session with Nessie/Iceberg/S3 configs.
2. Standardize the schema for 301.8M records.
3. Perform monthly batch ingestion to manage memory.
4. Verify metadata consistency in the Lakehouse.

## 🚀 Step 1: Configuration & Imports
We use the `org.apache.iceberg:iceberg-spark-runtime` and `nessie-spark-extensions` to enable the Medallion architecture.

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from datetime import datetime

# Production Credentials & Connection Details
NESSIE_URI = "http://140.238.224.207:19120/api/v1"
WAREHOUSE = "s3a://lakehouse-prod/warehouse"
AWS_ACCESS_KEY = "962c9f862226831e4edea90cfcfafb8a8dffcd51"
AWS_SECRET_KEY = "sd2rGU918DTmn35E4xJ8EV7BX2XUt7DkqC8v6WDNDUw="
AWS_S3_ENDPOINT = "https://bmcfe6z38foz.compat.objectstorage.ap-mumbai-1.oraclecloud.com"
AWS_REGION = "ap-mumbai-1"

print("✅ Configuration Loaded")

## 🛠️ Step 2: Initialize Spark Session
We use `HadoopFileIO` for the Bronze layer as it provides better stability during massive 300M+ record commits.

In [ ]:
spark = SparkSession.builder \
    .appName("Bronze-Ingestion-Interactive") \
    .config("spark.sql.shuffle.partitions", "1000") \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.3_2.12:1.3.1,org.projectnessie.nessie-integrations:nessie-spark-extensions-3.3_2.12:0.67.0,org.apache.hadoop:hadoop-aws:3.3.1") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,org.projectnessie.spark.extensions.NessieSparkSessionExtensions") \
    .config("spark.sql.catalog.nessie", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.nessie.uri", NESSIE_URI) \
    .config("spark.sql.catalog.nessie.ref", "bronze") \
    .config("spark.sql.catalog.nessie.authentication.type", "NONE") \
    .config("spark.sql.catalog.nessie.catalog-impl", "org.apache.iceberg.nessie.NessieCatalog") \
    .config("spark.sql.catalog.nessie.warehouse", WAREHOUSE) \
    .config("spark.sql.catalog.nessie.io-impl", "org.apache.iceberg.hadoop.HadoopFileIO") \
    .config("spark.hadoop.fs.s3a.access.key", AWS_ACCESS_KEY) \
    .config("spark.hadoop.fs.s3a.secret.key", AWS_SECRET_KEY) \
    .config("spark.hadoop.fs.s3a.endpoint", AWS_S3_ENDPOINT) \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

print("🚀 Spark Session Active!")

## 📖 Step 3: Source Data Discovery
We point to the raw Parquet repository on S3.

In [ ]:
RAW_DATA_PATH = "s3a://lakehouse-prod/raw/"
raw_df = spark.read.parquet(RAW_DATA_PATH)

print(f"🔍 Raw Schema Detected:")
raw_df.printSchema()

print("📊 Previewing first 5 records:")
raw_df.show(5)

## 🏗️ Step 4: Create Table & Namespace
We initialize the `ecommerce` namespace and the `orders_bronze` table.

In [ ]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.ecommerce")

# Create table if it doesn't exist
spark.sql("""
CREATE TABLE IF NOT EXISTS nessie.ecommerce.orders_bronze (
    event_time STRING,
    event_type STRING,
    product_id BIGINT,
    category_id BIGINT,
    category_code STRING,
    brand STRING,
    price DOUBLE,
    user_id BIGINT,
    user_session STRING
) USING iceberg
TBLPROPERTIES ('write.format.default'='parquet')
""")

print("✅ Bronze Table Ready in Nessie Catalog")

## 🔄 Step 5: Execute Monthly Batch Ingestion
To prevent OOM errors, we process one month at a time.

In [ ]:
months = [
    (2019, 12), (2020, 1), (2020, 2), (2020, 3), 
    (2020, 4), (2020, 5), (2020, 6), (2020, 7), 
    (2020, 8), (2020, 9), (2020, 10), (2020, 11)
]

for year, month in months:
    print(f"🔄 Processing Batch: {year}-{month:02d}")
    
    # filter raw data for the specific month via event_time string prefix
    batch_df = raw_df.filter(F.col("event_time").cast("string").startswith(f"{year}-{month:02d}"))
    
    # Append to Iceberg table
    batch_df.writeTo("nessie.ecommerce.orders_bronze").append()
    
    print(f"   ✓ Batch {year}-{month:02d} Ingested.")

print("\n🏆 FULL INGESTION COMPLETE")

## 📊 Step 6: Final Verification
Confirm the record count matches the production goal of ~301.8M.

In [ ]:
total_count = spark.table("nessie.ecommerce.orders_bronze").count()
print(f"📈 Final Bronze Count: {total_count:,} records")